In [1]:
# Task 0 — Install and prepare libraries
%pip -q install nltk spacy openpyxl

import re
import pandas as pd
import numpy as np
import nltk
import spacy
from collections import Counter
from IPython.display import display

# Download the NLTK resources used in this notebook
nltk_resources = [
    "punkt", "punkt_tab", "stopwords", "wordnet",
    "averaged_perceptron_tagger_eng", "maxent_ne_chunker_tab", "words"
]
for resource in nltk_resources:
    nltk.download(resource, quiet=True)

# Load spaCy English model; download it if it is not available
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
    nlp = spacy.load("en_core_web_sm")

print("Libraries and language resources are ready.")

Libraries and language resources are ready.


In [2]:
# Load the dataset
from google.colab import files
uploaded = files.upload()

file_name = next(iter(uploaded))
if file_name.lower().endswith((".xlsx", ".xls")):
    df = pd.read_excel(file_name)
elif file_name.lower().endswith(".csv"):
    df = pd.read_csv(file_name)
else:
    raise ValueError("Please upload an Excel (.xlsx) or CSV (.csv) file.")

required_columns = {"message_id", "message"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Keep the original columns unchanged
df["message"] = df["message"].fillna("").astype(str)
print("Dataset loaded successfully.")
display(df.head())

Saving nlp_customer_messages.xlsx to nlp_customer_messages (1).xlsx
Dataset loaded successfully.


,message_id,message
0,MSG001,"Good morning, My phone is not working :( Than..."
1,MSG002,"Hi, Thanks for your quick response! I need hel..."
2,MSG003,"Hi support, I LOVE this product 😍😍 Can someone..."
3,MSG004,"Hey there, The new update is AMAZING!! Please ..."
4,MSG005,"Dear team, The price is 250 EGP, too expensive..."


## Session 1 — Data inspection and regular expressions

### Task 1 — Load and inspect the dataset

In [3]:
print("First 5 messages:")
display(df.head())

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\\nMissing values:")
display(df.isna().sum().to_frame("missing_values"))

print("Duplicate message_id values:", df["message_id"].duplicated().sum())

df["char_count"] = df["message"].str.len()
longest_idx = df["char_count"].idxmax()
shortest_idx = df["char_count"].idxmin()

print("\\nLongest message:")
display(df.loc[[longest_idx], ["message_id", "message", "char_count"]])
print("Shortest message:")
display(df.loc[[shortest_idx], ["message_id", "message", "char_count"]])

print("\\nData quality summary:")
print(f"- Rows: {len(df)}")
print(f"- Missing message values: {df['message'].isna().sum()}")
print(f"- Duplicate IDs: {df['message_id'].duplicated().sum()}")
print(f"- Empty messages: {(df['message'].str.strip() == '').sum()}")
print("- Original message_id and message columns were not intentionally modified.")

First 5 messages:


,message_id,message
0,MSG001,"Good morning, My phone is not working :( Than..."
1,MSG002,"Hi, Thanks for your quick response! I need hel..."
2,MSG003,"Hi support, I LOVE this product 😍😍 Can someone..."
3,MSG004,"Hey there, The new update is AMAZING!! Please ..."
4,MSG005,"Dear team, The price is 250 EGP, too expensive..."


Shape: (80, 2)
Columns: ['message_id', 'message']
\nMissing values:


,missing_values
message_id,0
message,0


Duplicate message_id values: 0
\nLongest message:


,message_id,message,char_count
35,MSG036,"Hey there, I need help with payment for order ...",209


Shortest message:


,message_id,message,char_count
28,MSG029,"Dear team, Thanks! Great service :) Thank you.",46


\nData quality summary:
- Rows: 80
- Missing message values: 0
- Duplicate IDs: 0
- Empty messages: 0
- Original message_id and message columns were not intentionally modified.


### Task 2 — Extract information using regular expressions

In [4]:
EMAIL_PATTERN = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
URL_PATTERN = r"https?://[^\s<>()]+|www\.[^\s<>()]+"
PHONE_PATTERN = r"\b010\d{8}\b"
NUMBER_PATTERN = r"\b\d+(?:[.,]\d+)?\b"

def extract_patterns(message):
    text = str(message)
    return {
        "emails": re.findall(EMAIL_PATTERN, text, flags=re.IGNORECASE),
        "urls": re.findall(URL_PATTERN, text, flags=re.IGNORECASE),
        "phones": re.findall(PHONE_PATTERN, text),
        "numbers": re.findall(NUMBER_PATTERN, text),
        "has_urgent": bool(re.search(r"\burgent\b", text, flags=re.IGNORECASE))
    }

extracted = df["message"].apply(extract_patterns).apply(pd.Series)
df = pd.concat([df, extracted], axis=1)

print("Five messages containing extracted information:")
has_any = df[["emails", "urls", "phones", "numbers"]].apply(
    lambda col: col.map(bool)
).any(axis=1)
display(df.loc[has_any, ["message_id", "message", "emails", "urls", "phones", "numbers", "has_urgent"]].head(5))

print("\nMessage-level counts:")
print("Messages with email:", df["emails"].map(bool).sum())
print("Messages with URL:", df["urls"].map(bool).sum())
print("Messages with phone:", df["phones"].map(bool).sum())
print("Messages marked urgent:", df["has_urgent"].sum())

Five messages containing extracted information:


,message_id,message,emails,urls,phones,numbers,has_urgent
2,MSG003,"Hi support, I LOVE this product 😍😍 Can someone...",[customer3@example.com],[],[],[],False
3,MSG004,"Hey there, The new update is AMAZING!! Please ...",[],[https://help.example.com/ticket/10068],[],[10068],False
4,MSG005,"Dear team, The price is 250 EGP, too expensive...",[],[],[01010011705],"[250, 01010011705]",False
5,MSG006,"Dear support, The package for order 10102 is d...",[customer6@example.com],[],[],[10102],False
7,MSG008,"Hello team, Great quality, but delivery took 7...",[],[https://help.example.com/ticket/10136],[],"[7, 10136]",False



Message-level counts:
Messages with email: 26
Messages with URL: 20
Messages with phone: 16
Messages marked urgent: 11


### Task 3 — Clean messages while preserving useful text

In [5]:
def clean_message(message):
    text = str(message)
    # Replace specific patterns before general numbers
    text = re.sub(EMAIL_PATTERN, "[EMAIL]", text, flags=re.IGNORECASE)
    text = re.sub(URL_PATTERN, "[URL]", text, flags=re.IGNORECASE)
    text = re.sub(PHONE_PATTERN, "[PHONE]", text)
    text = re.sub(NUMBER_PATTERN, "[NUMBER]", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_message"] = df["message"].apply(clean_message)
display(df[["message_id", "message", "clean_message"]].head(10))

,message_id,message,clean_message
0,MSG001,"Good morning, My phone is not working :( Than...","Good morning, My phone is not working :( Thank..."
1,MSG002,"Hi, Thanks for your quick response! I need hel...","Hi, Thanks for your quick response! I need hel..."
2,MSG003,"Hi support, I LOVE this product 😍😍 Can someone...","Hi support, I LOVE this product 😍😍 Can someone..."
3,MSG004,"Hey there, The new update is AMAZING!! Please ...","Hey there, The new update is AMAZING!! Please ..."
4,MSG005,"Dear team, The price is 250 EGP, too expensive...","Dear team, The price is [NUMBER] EGP, too expe..."
5,MSG006,"Dear support, The package for order 10102 is d...","Dear support, The package for order [NUMBER] i..."
6,MSG007,"Hello, Bad quality... very disappointed. What ...","Hello, Bad quality... very disappointed. What ..."
7,MSG008,"Hello team, Great quality, but delivery took 7...","Hello team, Great quality, but delivery took [..."
8,MSG009,"Good morning, Thanks! Great service :) Thanks ...","Good morning, Thanks! Great service :) Thanks ..."
9,MSG010,"Hi, Can you change my delivery address? I need...","Hi, Can you change my delivery address? I need..."


### Task 4 — Test the extraction and cleaning functions

The test cases below contain an email, a phone number, a URL, and a message with none of these patterns. The regex rules are intentionally simple; unusual formats (international phone numbers, shortened URLs, or uncommon email formats) may not be detected.

In [6]:
test_messages = [
    "Please contact me at sara@example.com for details.",
    "Call me on 01012345678 tomorrow.",
    "Visit https://example.com/help for support.",
    "I really like the new service. Thank you!"
]

for i, message in enumerate(test_messages, start=1):
    print(f"\nTest {i}: {message}")
    print("Extracted:", extract_patterns(message))
    print("Cleaned:", clean_message(message))


Test 1: Please contact me at sara@example.com for details.
Extracted: {'emails': ['sara@example.com'], 'urls': [], 'phones': [], 'numbers': [], 'has_urgent': False}
Cleaned: Please contact me at [EMAIL] for details.

Test 2: Call me on 01012345678 tomorrow.
Extracted: {'emails': [], 'urls': [], 'phones': ['01012345678'], 'numbers': ['01012345678'], 'has_urgent': False}
Cleaned: Call me on [PHONE] tomorrow.

Test 3: Visit https://example.com/help for support.
Extracted: {'emails': [], 'urls': ['https://example.com/help'], 'phones': [], 'numbers': [], 'has_urgent': False}
Cleaned: Visit [URL] for support.

Test 4: I really like the new service. Thank you!
Extracted: {'emails': [], 'urls': [], 'phones': [], 'numbers': [], 'has_urgent': False}
Cleaned: I really like the new service. Thank you!


## Session 2 — Tokenization, normalization, stemming, lemmatization, stop words, POS and NER

### Task 5 — NLTK sentence and word tokenization

In [7]:
from nltk.tokenize import sent_tokenize, word_tokenize

df["sentences_nltk"] = df["clean_message"].apply(sent_tokenize)
df["tokens_nltk"] = df["clean_message"].apply(word_tokenize)
df["sentence_count"] = df["sentences_nltk"].apply(len)
df["token_count"] = df["tokens_nltk"].apply(len)

display(df[["message_id", "clean_message", "sentences_nltk", "tokens_nltk", "sentence_count", "token_count"]].head(10))

all_tokens = [token for row in df["tokens_nltk"] for token in row]
print("Top 5 most common tokens:")
display(pd.DataFrame(Counter(all_tokens).most_common(5), columns=["token", "count"]))

print("A token count can include punctuation, numbers, and symbols, while a word count usually counts only linguistic words.")

,message_id,clean_message,sentences_nltk,tokens_nltk,sentence_count,token_count
0,MSG001,"Good morning, My phone is not working :( Thank...","[Good morning, My phone is not working :( Than...","[Good, morning, ,, My, phone, is, not, working...",1,13
1,MSG002,"Hi, Thanks for your quick response! I need hel...","[Hi, Thanks for your quick response!, I need h...","[Hi, ,, Thanks, for, your, quick, response, !,...",3,15
2,MSG003,"Hi support, I LOVE this product 😍😍 Can someone...","[Hi support, I LOVE this product 😍😍 Can someon...","[Hi, support, ,, I, LOVE, this, product, 😍😍, C...",2,18
3,MSG004,"Hey there, The new update is AMAZING!! Please ...","[Hey there, The new update is AMAZING!!, Pleas...","[Hey, there, ,, The, new, update, is, AMAZING,...",3,17
4,MSG005,"Dear team, The price is [NUMBER] EGP, too expe...","[Dear team, The price is [NUMBER] EGP, too exp...","[Dear, team, ,, The, price, is, [, NUMBER, ], ...",3,23
5,MSG006,"Dear support, The package for order [NUMBER] i...","[Dear support, The package for order [NUMBER] ...","[Dear, support, ,, The, package, for, order, [...",3,27
6,MSG007,"Hello, Bad quality... very disappointed. What ...","[Hello, Bad quality... very disappointed., Wha...","[Hello, ,, Bad, quality, ..., very, disappoint...",3,17
7,MSG008,"Hello team, Great quality, but delivery took [...","[Hello team, Great quality, but delivery took ...","[Hello, team, ,, Great, quality, ,, but, deliv...",3,22
8,MSG009,"Good morning, Thanks! Great service :) Thanks ...","[Good morning, Thanks!, Great service :) Thank...","[Good, morning, ,, Thanks, !, Great, service, ...",3,24
9,MSG010,"Hi, Can you change my delivery address? I need...","[Hi, Can you change my delivery address?, I ne...","[Hi, ,, Can, you, change, my, delivery, addres...",3,22


Top 5 most common tokens:


,token,count
0,[,116
1,],116
2,",",92
3,.,89
4,!,88


A token count can include punctuation, numbers, and symbols, while a word count usually counts only linguistic words.


### Task 6 — Compare NLTK and spaCy tokenization

In [8]:
sample10 = df.head(10).copy()
sample10["tokens_spacy"] = sample10["clean_message"].apply(
    lambda text: [token.text for token in nlp(text)]
)
sample10["spacy_token_count"] = sample10["tokens_spacy"].apply(len)
sample10["nltk_token_count"] = sample10["tokens_nltk"].apply(len)

display(sample10[["message_id", "clean_message", "tokens_nltk", "tokens_spacy",
                  "nltk_token_count", "spacy_token_count"]])

# Show token attributes for the first message
doc = nlp(sample10.iloc[0]["clean_message"])
token_details = pd.DataFrame([
    {"token.text": token.text, "is_alpha": token.is_alpha, "is_punct": token.is_punct}
    for token in doc
])
display(token_details)

different = sample10[sample10["nltk_token_count"] != sample10["spacy_token_count"]]
print("Messages with different token counts:", len(different))
print("NLTK and spaCy may split contractions, punctuation, and special patterns differently.")

,message_id,clean_message,tokens_nltk,tokens_spacy,nltk_token_count,spacy_token_count
0,MSG001,"Good morning, My phone is not working :( Thank...","[Good, morning, ,, My, phone, is, not, working...","[Good, morning, ,, My, phone, is, not, working...",13,12
1,MSG002,"Hi, Thanks for your quick response! I need hel...","[Hi, ,, Thanks, for, your, quick, response, !,...","[Hi, ,, Thanks, for, your, quick, response, !,...",15,15
2,MSG003,"Hi support, I LOVE this product 😍😍 Can someone...","[Hi, support, ,, I, LOVE, this, product, 😍😍, C...","[Hi, support, ,, I, LOVE, this, product, 😍, 😍,...",18,19
3,MSG004,"Hey there, The new update is AMAZING!! Please ...","[Hey, there, ,, The, new, update, is, AMAZING,...","[Hey, there, ,, The, new, update, is, AMAZING,...",17,17
4,MSG005,"Dear team, The price is [NUMBER] EGP, too expe...","[Dear, team, ,, The, price, is, [, NUMBER, ], ...","[Dear, team, ,, The, price, is, [, NUMBER, ], ...",23,23
5,MSG006,"Dear support, The package for order [NUMBER] i...","[Dear, support, ,, The, package, for, order, [...","[Dear, support, ,, The, package, for, order, [...",27,27
6,MSG007,"Hello, Bad quality... very disappointed. What ...","[Hello, ,, Bad, quality, ..., very, disappoint...","[Hello, ,, Bad, quality, ..., very, disappoint...",17,16
7,MSG008,"Hello team, Great quality, but delivery took [...","[Hello, team, ,, Great, quality, ,, but, deliv...","[Hello, team, ,, Great, quality, ,, but, deliv...",22,22
8,MSG009,"Good morning, Thanks! Great service :) Thanks ...","[Good, morning, ,, Thanks, !, Great, service, ...","[Good, morning, ,, Thanks, !, Great, service, ...",24,23
9,MSG010,"Hi, Can you change my delivery address? I need...","[Hi, ,, Can, you, change, my, delivery, addres...","[Hi, ,, Can, you, change, my, delivery, addres...",22,22


,token.text,is_alpha,is_punct
0,Good,True,False
1,morning,True,False
2,",",False,True
3,My,True,False
4,phone,True,False
5,is,True,False
6,not,True,False
7,working,True,False
8,:(,False,True
9,Thanks,True,False


Messages with different token counts: 4
NLTK and spaCy may split contractions, punctuation, and special patterns differently.


### Task 7 — Lowercase normalization

In [9]:
df["lower_message"] = df["clean_message"].str.lower()
df["lower_tokens"] = df["lower_message"].apply(word_tokenize)

case_sensitive_unique = len(set(all_tokens))
lower_tokens_all = [token for row in df["lower_tokens"] for token in row]
lower_unique = len(set(lower_tokens_all))

print("Unique tokens before lowercasing:", case_sensitive_unique)
print("Unique tokens after lowercasing:", lower_unique)

# Find an actual token that appears in more than one capitalization, if present
token_forms = {}
for token in all_tokens:
    if token.isalpha():
        token_forms.setdefault(token.lower(), set()).add(token)
examples = [(k, sorted(v)) for k, v in token_forms.items() if len(v) > 1]
print("Example of case variants in this dataset:", examples[:5] if examples else "No case-variant example found.")
display(df[["message_id", "clean_message", "lower_message", "lower_tokens"]].head(5))

print("Lowercasing reduces vocabulary duplication, but it can remove capitalization clues such as names or acronyms.")

Unique tokens before lowercasing: 122
Unique tokens after lowercasing: 113
Example of case variants in this dataset: [('my', ['My', 'my']), ('phone', ['PHONE', 'phone']), ('not', ['NOT', 'not']), ('this', ['This', 'this']), ('can', ['Can', 'can'])]


,message_id,clean_message,lower_message,lower_tokens
0,MSG001,"Good morning, My phone is not working :( Thank...","good morning, my phone is not working :( thank...","[good, morning, ,, my, phone, is, not, working..."
1,MSG002,"Hi, Thanks for your quick response! I need hel...","hi, thanks for your quick response! i need hel...","[hi, ,, thanks, for, your, quick, response, !,..."
2,MSG003,"Hi support, I LOVE this product 😍😍 Can someone...","hi support, i love this product 😍😍 can someone...","[hi, support, ,, i, love, this, product, 😍😍, c..."
3,MSG004,"Hey there, The new update is AMAZING!! Please ...","hey there, the new update is amazing!! please ...","[hey, there, ,, the, new, update, is, amazing,..."
4,MSG005,"Dear team, The price is [NUMBER] EGP, too expe...","dear team, the price is [number] egp, too expe...","[dear, team, ,, the, price, is, [, number, ], ..."


Lowercasing reduces vocabulary duplication, but it can remove capitalization clues such as names or acronyms.


### Task 8 — Stemming with Porter, Snowball and Lancaster

In [10]:
from nltk.stem import PorterStemmer, SnowballStemmer, LancasterStemmer

porter = PorterStemmer()
snowball = SnowballStemmer("english")
lancaster = LancasterStemmer()

df["porter_stems"] = df["lower_tokens"].apply(lambda tokens: [porter.stem(t) for t in tokens])
df["snowball_stems"] = df["lower_tokens"].apply(lambda tokens: [snowball.stem(t) for t in tokens])
df["lancaster_stems"] = df["lower_tokens"].apply(lambda tokens: [lancaster.stem(t) for t in tokens])

# Compare 10 alphabetic words from the dataset
comparison_words = []
for token in lower_tokens_all:
    if token.isalpha() and token not in comparison_words:
        comparison_words.append(token)
    if len(comparison_words) == 10:
        break

stemming_table = pd.DataFrame({
    "original": comparison_words,
    "porter": [porter.stem(w) for w in comparison_words],
    "snowball": [snowball.stem(w) for w in comparison_words],
    "lancaster": [lancaster.stem(w) for w in comparison_words]
})
display(stemming_table)
display(df[["message_id", "porter_stems", "snowball_stems", "lancaster_stems"]].head(5))

print("Stemming removes word endings using rules and may produce non-dictionary forms.")
print("Lancaster is generally the most aggressive of these three stemmers.")

,original,porter,snowball,lancaster
0,good,good,good,good
1,morning,morn,morn,morn
2,my,my,my,my
3,phone,phone,phone,phon
4,is,is,is,is
5,not,not,not,not
6,working,work,work,work
7,thanks,thank,thank,thank
8,again,again,again,again
9,hi,hi,hi,hi


,message_id,porter_stems,snowball_stems,lancaster_stems
0,MSG001,"[good, morn, ,, my, phone, is, not, work, :, (...","[good, morn, ,, my, phone, is, not, work, :, (...","[good, morn, ,, my, phon, is, not, work, :, (,..."
1,MSG002,"[hi, ,, thank, for, your, quick, respons, !, i...","[hi, ,, thank, for, your, quick, respons, !, i...","[hi, ,, thank, for, yo, quick, respons, !, i, ..."
2,MSG003,"[hi, support, ,, i, love, thi, product, 😍😍, ca...","[hi, support, ,, i, love, this, product, 😍😍, c...","[hi, support, ,, i, lov, thi, produc, 😍😍, can,..."
3,MSG004,"[hey, there, ,, the, new, updat, is, amaz, !, ...","[hey, there, ,, the, new, updat, is, amaz, !, ...","[hey, ther, ,, the, new, upd, is, amaz, !, !, ..."
4,MSG005,"[dear, team, ,, the, price, is, [, number, ], ...","[dear, team, ,, the, price, is, [, number, ], ...","[dear, team, ,, the, pric, is, [, numb, ], egp..."


Stemming removes word endings using rules and may produce non-dictionary forms.
Lancaster is generally the most aggressive of these three stemmers.


### Task 9 — Lemmatization with NLTK WordNet and spaCy

In [11]:
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

lemmatizer = WordNetLemmatizer()

def nltk_pos_to_wordnet(tag):
    if tag.startswith("J"): return wordnet.ADJ
    if tag.startswith("V"): return wordnet.VERB
    if tag.startswith("N"): return wordnet.NOUN
    if tag.startswith("R"): return wordnet.ADV
    return wordnet.NOUN

def nltk_lemmatize_tokens(tokens):
    tagged = nltk.pos_tag(tokens)
    return [lemmatizer.lemmatize(word, nltk_pos_to_wordnet(tag)) for word, tag in tagged]

df["lemmas_nltk"] = df["lower_tokens"].apply(nltk_lemmatize_tokens)
df["lemmas_spacy"] = df["clean_message"].apply(
    lambda text: [token.lemma_ for token in nlp(text)]
)

lemma_words = comparison_words[:10]
lemma_table = pd.DataFrame({
    "original": lemma_words,
    "porter_stem": [porter.stem(w) for w in lemma_words],
    "nltk_lemma": [lemmatizer.lemmatize(w) for w in lemma_words],
    "spacy_lemma": [token.lemma_ for w in lemma_words for token in nlp(w)][:len(lemma_words)]
})
display(lemma_table)
display(df[["message_id", "lemmas_nltk", "lemmas_spacy"]].head(5))

print("Lemmatization uses vocabulary and grammatical information to return a base word.")
print("POS information matters because the same word can have different lemmas depending on its grammatical role.")

,original,porter_stem,nltk_lemma,spacy_lemma
0,good,good,good,good
1,morning,morn,morning,morning
2,my,my,my,my
3,phone,phone,phone,phone
4,is,is,is,be
5,not,not,not,not
6,working,work,working,work
7,thanks,thank,thanks,thank
8,again,again,again,again
9,hi,hi,hi,hi


,message_id,lemmas_nltk,lemmas_spacy
0,MSG001,"[good, morning, ,, my, phone, be, not, work, :...","[good, morning, ,, my, phone, be, not, work, :..."
1,MSG002,"[hi, ,, thanks, for, your, quick, response, !,...","[hi, ,, thank, for, your, quick, response, !, ..."
2,MSG003,"[hi, support, ,, i, love, this, product, 😍😍, c...","[hi, support, ,, I, love, this, product, 😍, 😍,..."
3,MSG004,"[hey, there, ,, the, new, update, be, amazing,...","[hey, there, ,, the, new, update, be, amazing,..."
4,MSG005,"[dear, team, ,, the, price, be, [, number, ], ...","[dear, team, ,, the, price, be, [, number, ], ..."


Lemmatization uses vocabulary and grammatical information to return a base word.
POS information matters because the same word can have different lemmas depending on its grammatical role.


### Task 10 — Stop-word removal and negation check

In [12]:
from nltk.corpus import stopwords

nltk_stop_words = set(stopwords.words("english"))
spacy_stop_words = nlp.Defaults.stop_words

df["tokens_no_stop_nltk"] = df["lower_tokens"].apply(
    lambda tokens: [t for t in tokens if t.lower() not in nltk_stop_words]
)
df["tokens_no_stop_spacy"] = df["lower_tokens"].apply(
    lambda tokens: [t for t in tokens if t.lower() not in spacy_stop_words]
)

df["nltk_stop_count"] = df["tokens_no_stop_nltk"].apply(len)
df["spacy_stop_count"] = df["tokens_no_stop_spacy"].apply(len)
display(df[["message_id", "lower_tokens", "tokens_no_stop_nltk",
            "tokens_no_stop_spacy", "nltk_stop_count", "spacy_stop_count"]].head(10))

negations = {"not", "no", "never"}
print("Negation words in NLTK stop words:", sorted(negations & nltk_stop_words))
print("Negation words in spaCy stop words:", sorted(negations & spacy_stop_words))

# Safer policy: remove stop words but preserve negations
df["tokens_no_stop_keep_negation"] = df["lower_tokens"].apply(
    lambda tokens: [t for t in tokens
                    if t.lower() not in (nltk_stop_words - negations)]
)
print("Negation-aware example:")
display(df[["message_id", "lower_tokens", "tokens_no_stop_keep_negation"]].head(5))

print("Removing negation words can reverse sentiment or intent. A safer policy is to keep not, no, and never.")

,message_id,lower_tokens,tokens_no_stop_nltk,tokens_no_stop_spacy,nltk_stop_count,spacy_stop_count
0,MSG001,"[good, morning, ,, my, phone, is, not, working...","[good, morning, ,, phone, working, :, (, thank...","[good, morning, ,, phone, working, :, (, thank...",9,9
1,MSG002,"[hi, ,, thanks, for, your, quick, response, !,...","[hi, ,, thanks, quick, response, !, need, help...","[hi, ,, thanks, quick, response, !, need, help...",12,12
2,MSG003,"[hi, support, ,, i, love, this, product, 😍😍, c...","[hi, support, ,, love, product, 😍😍, someone, h...","[hi, support, ,, love, product, 😍😍, help, ?, e...",14,13
3,MSG004,"[hey, there, ,, the, new, update, is, amazing,...","[hey, ,, new, update, amazing, !, !, please, c...","[hey, ,, new, update, amazing, !, !, check, .,...",14,13
4,MSG005,"[dear, team, ,, the, price, is, [, number, ], ...","[dear, team, ,, price, [, number, ], egp, ,, e...","[dear, team, ,, price, [, number, ], egp, ,, e...",17,16
5,MSG006,"[dear, support, ,, the, package, for, order, [...","[dear, support, ,, package, order, [, number, ...","[dear, support, ,, package, order, [, number, ...",21,20
6,MSG007,"[hello, ,, bad, quality, ..., very, disappoint...","[hello, ,, bad, quality, ..., disappointed, .,...","[hello, ,, bad, quality, ..., disappointed, .,...",12,12
7,MSG008,"[hello, team, ,, great, quality, ,, but, deliv...","[hello, team, ,, great, quality, ,, delivery, ...","[hello, team, ,, great, quality, ,, delivery, ...",21,20
8,MSG009,"[good, morning, ,, thanks, !, great, service, ...","[good, morning, ,, thanks, !, great, service, ...","[good, morning, ,, thanks, !, great, service, ...",21,21
9,MSG010,"[hi, ,, can, you, change, my, delivery, addres...","[hi, ,, change, delivery, address, ?, need, he...","[hi, ,, change, delivery, address, ?, need, he...",16,15


Negation words in NLTK stop words: ['no', 'not']
Negation words in spaCy stop words: ['never', 'no', 'not']
Negation-aware example:


,message_id,lower_tokens,tokens_no_stop_keep_negation
0,MSG001,"[good, morning, ,, my, phone, is, not, working...","[good, morning, ,, phone, not, working, :, (, ..."
1,MSG002,"[hi, ,, thanks, for, your, quick, response, !,...","[hi, ,, thanks, quick, response, !, need, help..."
2,MSG003,"[hi, support, ,, i, love, this, product, 😍😍, c...","[hi, support, ,, love, product, 😍😍, someone, h..."
3,MSG004,"[hey, there, ,, the, new, update, is, amazing,...","[hey, ,, new, update, amazing, !, !, please, c..."
4,MSG005,"[dear, team, ,, the, price, is, [, number, ], ...","[dear, team, ,, price, [, number, ], egp, ,, e..."


Removing negation words can reverse sentiment or intent. A safer policy is to keep not, no, and never.


### Task 11 — Part-of-speech (POS) tagging

In [14]:
# POS tagging uses readable, lowercased text (not stemmed text)
# We must pass tokenized lists (from lower_tokens) to nltk.pos_tag rather than raw strings
df["pos_tags"] = df["lower_tokens"].apply(nltk.pos_tag)

tag_counts = Counter(tag for row in df["pos_tags" ] for _, tag in row)
print("POS tag counts:")
display(pd.DataFrame(tag_counts.most_common(), columns=["POS_tag", "count"]))

verbs = Counter(word for row in df["pos_tags"] for word, tag in row
                if tag.startswith("VB") and word.isalpha())
adjectives = Counter(word for row in df["pos_tags"] for word, tag in row
                     if tag.startswith("JJ") and word.isalpha())

print("Top 5 verbs:")
display(pd.DataFrame(verbs.most_common(5), columns=["verb", "count"]))
print("Top 5 adjectives:")
display(pd.DataFrame(adjectives.most_common(5), columns=["adjective", "count"]))

display(df[["message_id", "lower_message", "pos_tags"]].head(5))
print("POS tagging assigns grammatical roles such as noun, verb, adjective, and adverb based on context.")

POS tag counts:


,POS_tag,count
0,NN,505
1,.,205
2,JJ,194
3,VB,119
4,NNP,105
5,",",92
6,RB,72
7,PRP,61
8,:,52
9,IN,48


Top 5 verbs:


,verb,count
0,is,28
1,help,25
2,i,24
3,please,20
4,call,16


Top 5 adjectives:


,adjective,count
0,url,20
1,dear,20
2,great,12
3,urgent,11
4,good,10


,message_id,lower_message,pos_tags
0,MSG001,"good morning, my phone is not working :( thank...","[(good, JJ), (morning, NN), (,, ,), (my, PRP$)..."
1,MSG002,"hi, thanks for your quick response! i need hel...","[(hi, NN), (,, ,), (thanks, NNS), (for, IN), (..."
2,MSG003,"hi support, i love this product 😍😍 can someone...","[(hi, JJ), (support, NN), (,, ,), (i, VB), (lo..."
3,MSG004,"hey there, the new update is amazing!! please ...","[(hey, NN), (there, RB), (,, ,), (the, DT), (n..."
4,MSG005,"dear team, the price is [number] egp, too expe...","[(dear, JJ), (team, NN), (,, ,), (the, DT), (p..."


POS tagging assigns grammatical roles such as noun, verb, adjective, and adverb based on context.


### Task 12 — Named Entity Recognition (NER) with spaCy

In [15]:
# NER is applied to the original readable message, not the cleaned or stemmed text
ner_records = []
for _, row in df.head(15).iterrows():
    doc = nlp(row["message"])
    for ent in doc.ents:
        ner_records.append({
            "message_id": row["message_id"],
            "entity_text": ent.text,
            "entity_label": ent.label_
        })

entities_df = pd.DataFrame(ner_records, columns=["message_id", "entity_text", "entity_label"])
print("Entities found in the first 15 messages:")
display(entities_df)

print("Entity counts by label:")
if not entities_df.empty:
    display(entities_df["entity_label"].value_counts().rename_axis("entity_label").reset_index(name="count"))
else:
    print("No named entities were detected in the first 15 messages.")

print("NER examples grouped by message:")
for message_id in df.head(5)["message_id"]:
    print(message_id, "->", entities_df.loc[entities_df["message_id"] == message_id, "entity_text"].tolist())

print("General-purpose NER is not guaranteed to recognize emails, phone numbers, or order IDs.")
print("Regex is usually more reliable for structured patterns such as email addresses and phone numbers.")

Entities found in the first 15 messages:


,message_id,entity_text,entity_label
0,MSG005,250,CARDINAL
1,MSG005,EGP,ORG
2,MSG005,01010011705,DATE
3,MSG006,10102,DATE
4,MSG007,@support,GPE
5,MSG008,7 days,DATE
6,MSG010,01010023410,DATE
7,MSG011,A-45-2026,DATE
8,MSG013,2,CARDINAL
9,MSG013,1,CARDINAL


Entity counts by label:


,entity_label,count
0,DATE,6
1,CARDINAL,4
2,GPE,2
3,ORG,1


NER examples grouped by message:
MSG001 -> []
MSG002 -> []
MSG003 -> []
MSG004 -> []
MSG005 -> ['250', 'EGP', '01010011705']
General-purpose NER is not guaranteed to recognize emails, phone numbers, or order IDs.
Regex is usually more reliable for structured patterns such as email addresses and phone numbers.


## Final review and export

The original `message_id` and `message` columns remain available. All processing results were added as new columns.

In [16]:
print("Final dataset shape:", df.shape)
print("Final columns:")
print(df.columns.tolist())

# Save processed results for reference
df.to_csv("nlp_customer_messages_processed.csv", index=False)
entities_df.to_csv("nlp_entities_first_15_messages.csv", index=False)

print("Saved: nlp_customer_messages_processed.csv")
print("Saved: nlp_entities_first_15_messages.csv")

Final dataset shape: (80, 26)
Final columns:
['message_id', 'message', 'char_count', 'emails', 'urls', 'phones', 'numbers', 'has_urgent', 'clean_message', 'sentences_nltk', 'tokens_nltk', 'sentence_count', 'token_count', 'lower_message', 'lower_tokens', 'porter_stems', 'snowball_stems', 'lancaster_stems', 'lemmas_nltk', 'lemmas_spacy', 'tokens_no_stop_nltk', 'tokens_no_stop_spacy', 'nltk_stop_count', 'spacy_stop_count', 'tokens_no_stop_keep_negation', 'pos_tags']
Saved: nlp_customer_messages_processed.csv
Saved: nlp_entities_first_15_messages.csv
